# MCL repair model: fine-tuning on Google Colab

Run the cells **one at a time, top to bottom** (click a cell, press Shift+Enter).
Wait for each cell to finish (the circle spinner at its left stops) before running the next.

**Before you start:** Runtime menu > *Change runtime type* > choose **T4 GPU** > Save.

## Step 1: Check the GPU is on

In [ ]:
!nvidia-smi

You should see a table that mentions a Tesla T4. If it says *command not found*, the GPU is off. Fix the runtime type above.

## Step 2: Install the two extra libraries

In [ ]:
!pip install -q peft accelerate

## Step 3: Upload the project
Run the cell, click *Choose files*, and pick **lexis_phase3.zip**.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip -q -o lexis_phase3.zip -d /content
%cd /content/lexis
!ls

## Step 4: Check the compiler still passes its tests in Colab

In [ ]:
!python -m pytest tests -q

Expect a line ending in **passed**. If anything fails, stop and send me the message.

## Step 5: Connect Google Drive (so the trained model is not lost)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 6: Smoke test (about 1-3 minutes)
This trains on only 64 examples. It proves the whole setup works before the long run.

In [ ]:
!python future_work/train_lora.py --smoke --batch 1 --grad-accum 16 --out /content/smoke_adapter

Look for **Adapter saved to: /content/smoke_adapter** at the end. If you see a red error instead, copy the last 20 lines and send them to me.

## Step 7: Real training
Uses 1,500 examples for 2 epochs. This is the long step. Keep the tab open.

In [ ]:
!python future_work/train_lora.py --batch 1 --grad-accum 16 --out /content/mcl_adapter

In [ ]:
# copy the trained adapter to Drive so it survives if Colab disconnects
!cp -r /content/mcl_adapter /content/drive/MyDrive/mcl_adapter

## Step 8: Evaluate the original model (before fine-tuning)
This is the baseline to beat. It uses the first 150 test examples. Step 9 must use the **same** number so the comparison is fair. Remove `--limit 150` in both cells to use all 408 (slower).

In [ ]:
!python evaluate_model.py --backend hf --label base --limit 150 --out results/base.json

## Step 9: Evaluate your fine-tuned model

In [ ]:
!python evaluate_model.py --backend hf --label finetuned --adapter /content/mcl_adapter --limit 150 --attempts 2 --out results/finetuned.json

## Step 10: Side-by-side comparison
This table is the main result for your report.

In [ ]:
!python compare_results.py results/base.json results/finetuned.json

## Step 11: Try it on a broken program

In [ ]:
!python mcl_repair.py tests/programs/malformed_multi_error.mcl --method hf --adapter /content/mcl_adapter

## Step 12: Download the results
This saves the numbers, the training log and the adapter to your computer.

In [ ]:
!cp -r /content/mcl_adapter .
!zip -qr results_and_adapter.zip results mcl_adapter
files.download('results_and_adapter.zip')